# Table of contents

* **Introduction**
* **Objective**
* **Recurrent Neural Networks (RNNs) Foundations**
* **Data Dictionary**
* **Pipeline Overview**
* **Approach**
* **Setup Environments**
* **Data Loading**
* **Exploratory Data Analysis (EDA)**
* **Features Engineering**
* **Model Training**
    * Model Architecture and Model Building (Keras)
    * Model Training (Keras)
    * Model Architecture and Model Building (TensorFlow custom functions)
    * Model Training (TensorFlow custom functions)
* **Model Evaluation**
* **Conclusion**
* **References**

# Introduction

This project explores Recurrent Neural Networks (RNNs) for financial time series prediction. Unlike standard neural networks, RNNs are architecturally designed for sequential data, processing inputs iteratively while maintaining a hidden state that acts as memory from prior time steps. Specifically, the task involves predicting the next day's price features (open, close, low, high, and volume) for the stock EQIX based on a 30-day lookback window. The implementation uses the flexible, high-level API of Keras within the TensorFlow framework, emphasizing best practices in data handling and model regularization.

# Objective

The primary objective is to build a robust model capable of multi-step-ahead forecasting (predicting five correlated output features simultaneously) by leveraging sequential dependencies. This requires solving the inherent challenges of deep RNNs, namely the vanishing gradient problem, which hinders the capture of long-term dependencies across the 30-day sequence. The project specifically utilizes two RNN model variants—a standard Stacked SimpleRNN and a reusable Custom RNN Block—to compare architectural design and ensure training stability through techniques like Layer Normalization and Gradient Clipping.

# Recurrent Neural Networks (RNNs) Foundations

Recurrent Neural Networks (RNNs) are a class of artificial neural networks specifically designed to process sequential data, such as time series, speech, and text. Unlike standard feedforward networks, which treat each input independently, RNNs possess an internal "memory" that allows information from previous steps to persist and influence the processing of the current input. This characteristic makes them the foundational architecture for sequence modeling.

**Core Principles**

**1. Sequential Processing and Recurrence**

The defining feature of an RNN is its recurrent loop. At any time step $t$, the network receives two inputs: the current data point in the sequence, $\mathbf{x}_t$, and the hidden state from the previous time step, $\mathbf{h}_{t-1}$.

The new hidden state, $\mathbf{h}_t$, is calculated using a non-linear activation function (like $\tanh$) and a set of shared weights, $W_{xh}$ for the input and $W_{hh}$ for the recurrent connection:


$$\mathbf{h}_t = f(W_{xh}\mathbf{x}_t + W_{hh}\mathbf{h}_{t-1} + \mathbf{b}_h)$$

**2. Weight Sharing Across Time**

A crucial concept is weight sharing. The same weight matrices ($W_{xh}$, $W_{hh}$, and the output weight $W_{hy}$) are used for every time step in the sequence. This greatly reduces the number of parameters the network must learn and allows the model to generalize across the entire sequence length, learning a single function that processes all temporal steps.

**3. Backpropagation Through Time (BPTT)**

Training an RNN involves an adaptation of the standard backpropagation algorithm called Backpropagation Through Time (BPTT). The recurrence relation is unrolled over the entire sequence, treating the network at each time step as a separate layer in a deep feedforward network. Gradients are then calculated and propagated backward through this unrolled structure.

**Inherent Challenges (The Gradient Problem)**

While powerful for short sequences, the vanilla RNN architecture suffers from fundamental problems during BPTT:

**1. Vanishing Gradient Problem**

As the gradient is repeatedly multiplied by the weight matrices through many time steps (layers), the value of the gradient can shrink exponentially towards zero. This makes the gradient contribution from distant past inputs (early steps in the sequence) negligible. Consequently, the network struggles to learn long-term dependencies—a critical limitation for tasks like complex language modeling or predicting long financial trends.

2. Exploding Gradient Problem

Conversely, if the weights are large, the gradient can grow exponentially, leading to an exploding gradient. This results in numerical instability, causing large updates to the model parameters, which renders the training unstable or ineffective. This is typically addressed using gradient clipping, a technique that scales down the gradients if they exceed a certain threshold.

**Solutions and Successors**

The severe limitations of the vanilla RNN led to the development of more sophisticated architectures that include mechanisms for regulating the flow of information:
- Long Short-Term Memory (LSTM): Introduces specialized gates (input, forget, and output) and a cell state to explicitly control which information is stored, updated, or discarded. LSTMs effectively solve the vanishing gradient problem and are the standard choice for most sequence tasks.
- Gated Recurrent Unit (GRU): A simplified version of the LSTM that merges the cell state and hidden state, using only two gates (reset and update). GRUs are often computationally less expensive and perform comparably to LSTMs on many datasets.

The general RNN structure serves as the conceptual basis, but modern time series and sequence modeling almost exclusively rely on LSTMs or GRUs due to their superior ability to capture information over long spans of time.

# Data Dictionary

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-za14{border-color:inherit;text-align:left;vertical-align:bottom}
.tg .tg-7zrl{text-align:left;vertical-align:bottom}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-za14">Field Name</th>
    <th class="tg-7zrl">Description</th>
    <th class="tg-7zrl">Data Type</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-7zrl">date</td>
    <td class="tg-7zrl">The date range for each data entry. This appears to be a combined string of start and end dates.</td>
    <td class="tg-7zrl">Text</td>
  </tr>
  <tr>
    <td class="tg-7zrl">DateTimeCount</td>
    <td class="tg-7zrl">The count of DateTime entries within the specified date range.</td>
    <td class="tg-7zrl">Numeric (Integer)</td>
  </tr>
  <tr>
    <td class="tg-7zrl">symbol</td>
    <td class="tg-7zrl">An identifier or code, possibly for a financial instrument or category.</td>
    <td class="tg-7zrl">Text</td>
  </tr>
  <tr>
    <td class="tg-7zrl">open</td>
    <td class="tg-7zrl">The opening price/value for a given period.</td>
    <td class="tg-7zrl">Numeric (Float)</td>
  </tr>
  <tr>
    <td class="tg-7zrl">close</td>
    <td class="tg-7zrl">The closing price/value for a given period.</td>
    <td class="tg-7zrl">Numeric (Float)</td>
  </tr>
  <tr>
    <td class="tg-7zrl">low</td>
    <td class="tg-7zrl">The lowest price/value recorded within a given period.</td>
    <td class="tg-7zrl">Numeric (Float)</td>
  </tr>
  <tr>
    <td class="tg-7zrl">high</td>
    <td class="tg-7zrl">The highest price/value recorded within a given period.</td>
    <td class="tg-7zrl">Numeric (Float)</td>
  </tr>
  <tr>
    <td class="tg-7zrl">volume</td>
    <td class="tg-7zrl">The trading volume (number of units traded) within a given period.</td>
    <td class="tg-7zrl">Numeric (Integer)</td>
  </tr>
</tbody></table>

**[Dataset Link](https://www.kaggle.com/datasets/dgawlik/nyse)**

# Pipeline Overview

The forecasting process is structured into a linear pipeline ensuring data integrity and effective model optimization:

- Data Acquisition and Filtering: Isolating the target time series (EQIX).
- Chronological Splitting: Partitioning the data into train, validation, and test sets to simulate real-world prediction.
- Data Preprocessing: Feature selection, scaling (normalization), and transformation.
- Sequence Generation: Creating the 3D-input tensors required by RNNs.
- Model Building: Defining and compiling the Keras models.
- Training and Regularization: Optimization using callbacks like Early Stopping.
- Evaluation: Calculating error metrics and visualizing forecasts on the original price scale.

# Approach

**Preprocess Pipeline Approach**

Effective preprocessing is mandatory for stable RNN training:
- Chronological Train/Test Split: Time series data must be split by date to prevent data leakage, where the model could learn from future information. An $80/20$ split ensures adequate training history, followed by an internal $20%$ validation split from the training data for monitoring.
- Feature Selection and Scaling: Only the numerical price and volume features are retained. MinMaxScaler is applied, fitting the scaling parameters exclusively on the training data. Scaling features to a range (e.g., $0$ to $1$) stabilizes gradient computations, particularly within the $\tanh$ activation functions typical of RNN layers.
- Sequence Creation: The data is restructured from 2D tabular form to the 3D format required by Keras RNN layers: (samples, time_steps, features). A 30-day lookback window (time_steps = 30) is used to predict the 5 features of the subsequent day. Overlapping windows are employed to maximize the number of training samples derived from the limited time series.
- Tensor Conversion: Finalized NumPy arrays are converted to TensorFlow constant tensors to leverage optimized graph operations and GPU acceleration during training.

**Model Training Approach**

The training process emphasizes robust optimization and regularization to counter overfitting:
- SimpleRNN Architecture: The model uses stacked SimpleRNN layers. The return_sequences=True argument must be used for intermediate layers to pass the sequence output to the next RNN layer. The final RNN layer uses return_sequences=False to pass only the last hidden state of the sequence to the final dense layer for prediction.
- Initialization and Normalization: Layer Normalization is applied immediately to the input sequence to standardize feature distributions and stabilize internal layer activations. Weights are initialized using Glorot Uniform (for dense layers) and Orthogonal (for RNN kernels), which are known to improve convergence.
- Optimization and Clipping: The Adam optimizer is utilized, starting with a reduced learning rate (e.g., $1\text{e-}3$) to prevent early divergence. Crucially, Gradient Clipping (clipnorm=1.0) is implemented to address the exploding gradient problem common in RNNs, which limits the magnitude of gradient updates.
- Regularization Callbacks:
    - Early Stopping (patience=20) monitors the validation loss and halts training when improvement stalls, restoring the best weights to prevent overfitting.
    - ReduceLROnPlateau monitors the validation loss and automatically reduces the learning rate when no improvement is detected, helping the optimizer escape local minima and achieve finer convergence.

**Model Evaluation Approach**

Model performance is assessed by moving predictions back to the original price scale and examining both error magnitude and directional accuracy.
- Prediction and Inverse Scaling: The final model is set to inference mode, and predictions are generated for all three datasets. The predictions and corresponding true targets are then transformed back to the original dollar-price scale using the stored MinMaxScaler inverse function.
- Regression Metrics (RMSE): The Root Mean Squared Error (RMSE) is calculated on the inverse-scaled open price feature across the train, validation, and test sets. This provides an easily interpretable measure of the average dollar-value prediction error. The test RMSE serves as the unbiased performance estimate.
- Directional Accuracy: A financially relevant metric, Sign Accuracy, is calculated by comparing the predicted direction of change (close - open) to the actual direction of change. A model must achieve significantly better than $50%$ accuracy to demonstrate real forecasting skill.
- Visualization: Forecasts are plotted against actual prices across the entire historical timeline, with a zoomed view on the test set. This visual analysis helps diagnose the model's behavior, particularly its ability to track volatility versus simply predicting the mean.

# Setup Environments

**Import Libraries**

In [ ]:
import numpy as np
import pandas as pd
import math
import sklearn
import sklearn.preprocessing
import datetime
import os
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
import warnings
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings("ignore", category=FutureWarning, module='seaborn._oldcore')

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")

# Data Loading

The process begins with loading the New York Stock Exchange (NYSE) data, which contains various features for multiple stocks over time. The primary challenge in time series analysis is maintaining the chronological integrity of the data to accurately simulate real-world forecasting.

In [ ]:
df = pd.read_csv('/kaggle/input/nyse/prices-split-adjusted.csv')
df.head()

**Train and Test Split**

The dataset is initially loaded, and its sheer size necessitates an efficient chronological split into training (80%) and testing (20%) subsets. The data must be sorted strictly by the $\mathbf{date}$ column before splitting. This non-randomized partitioning ensures the training process uses only historical information, and the test set represents the genuinely unseen future data. Violating this constraint would lead to data leakage, resulting in an overly optimistic and unreliable model evaluation. The shapes of the resulting subsets confirm the non-random, time-based division.

In [ ]:
df = df.sort_values(by='date', ascending=True).reset_index(drop=True)

test_size_ratio = 0.2 # 20% for test set
split_point = int(len(df) * (1 - test_size_ratio))

train = df.iloc[:split_point].copy()
test = df.iloc[split_point:].copy()

print("\nData split chronologically by 'date' column:")
print(f"Train DataFrame shape: {train.shape}")
print(f"Test DataFrame shape: {test.shape}")

In [ ]:
train = train.sort_values(by='date').reset_index(drop=True)
test = test.sort_values(by='date').reset_index(drop=True)

**Stock Selection and Filtering**

For concentrated, single-series modeling, the data is filtered to isolate only the records corresponding to the target stock, $\mathbf{EQIX}$. Focusing on a single ticker simplifies the initial multivariate modeling problem, ensuring that the recurrent model learns the specific temporal patterns and volatility characteristics unique to that equity.

In [ ]:
train = train[train.symbol == 'EQIX'].copy()
test = test[test.symbol == 'EQIX'].copy()
# Sort train dataframe based on date column from less value to more value
train = train.sort_values(by='date').reset_index(drop=True)
test = test.sort_values(by='date').reset_index(drop=True)
train

# Exploratory Data Analysis (EDA)

Exploratory analysis provides critical insights into data trends, volatility, and the internal relationships between features, informing the subsequent feature engineering and model design phases.

**Basic Statistics**

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

**Features Trends and Stationarity**

The price features ($\mathbf{open, close, low, high}$) display a strong upward, non-stationary trend over the observation period, reflecting significant capital appreciation. The lines remain tightly grouped, indicating low intraday volatility in most periods. The $\mathbf{volume}$ feature, however, is characterized by large, intermittent spikes, suggesting irregular trading activity driven by news or corporate events. Time series models must account for these non-stationary trends and volatility bursts.

In [ ]:
train['time_idx'] = range(len(train))

ctxs_data = train[train.symbol == 'EQIX']

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.lineplot(x='time_idx', y='open', data=ctxs_data, color='red', label='Open', ax=axes[0])
sns.lineplot(x='time_idx', y='close', data=ctxs_data, color='green', label='Close', ax=axes[0])
sns.lineplot(x='time_idx', y='low', data=ctxs_data, color='blue', label='Low', ax=axes[0])
sns.lineplot(x='time_idx', y='high', data=ctxs_data, color='black', label='High', ax=axes[0])

axes[0].set_title('Stock Price for EQIX', fontsize=14)
axes[0].set_xlabel('Time [days]', fontsize=12)
axes[0].set_ylabel('Price', fontsize=12)
axes[0].legend(loc='best')
axes[0].grid(True, linestyle='--', alpha=0.7)

sns.lineplot(x='time_idx', y='volume', data=ctxs_data, color='purple', label='Volume', ax=axes[1])

axes[1].set_title('Stock Volume for CTXS', fontsize=14)
axes[1].set_xlabel('Time [days]', fontsize=12)
axes[1].set_ylabel('Volume', fontsize=12)
axes[1].legend(loc='best')
axes[1].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

**Correlation Analysis**

Feature correlation analysis, presented in the heatmap, reveals critical statistical relationships within the time series data.

In [ ]:
train = train.drop(columns=['symbol'])
test = test.drop(columns=['symbol'])
train

In [ ]:
train['date'] = pd.to_datetime(train['date'])

train['year'] = train['date'].dt.year
train['month'] = train['date'].dt.month
train['day'] = train['date'].dt.day
train

In [ ]:
numerical_cols = ['year','month','day', 'open', 'close', 'low', 'high', 'volume']

correlation_matrix = train[numerical_cols].corr(method='pearson')

plt.figure(figsize=(20, 15))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='rocket',
    fmt=".2f",
    linewidths=.5
)
plt.title('Correlation Matrix of Stock Numerical Features', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

**Correlation Analysis Interpretation**

- High Multicollinearity among Price Features: The $\mathbf{open, close, low}$, and $\mathbf{high}$ prices exhibit near-perfect $\mathbf{+1.00}$ correlation coefficients. This is expected in financial data, as the day's prices are tightly bound. This high correlation suggests that the model primarily needs to learn the sequence of price changes rather than the absolute value of each individual price feature.
- Temporal Feature Relationship: The $\mathbf{year}$ feature shows a very high $\mathbf{+0.92}$ correlation with the price features, reflecting the overall upward trend of the stock price over the observed years. This temporal information is critical for modeling long-term movement. $\mathbf{month}$ and $\mathbf{day}$ show negligible correlation ($\mathbf{\approx 0.00}$) with the price features, indicating they provide little linear predictive value on their own but may capture cyclical monthly or weekly effects when processed by a non-linear network like the RNN.
- Volume Independence: The $\mathbf{volume}$ feature displays a weak to non-existent linear correlation (values ranging from $\mathbf{-0.09}$ to $\mathbf{0.01}$) with all price and temporal features. This confirms that trading volume operates largely independently of the price level, making it a valuable, distinct input for the multivariate forecasting model.

# Features Engineering

The goal of this phase is to refine the features, ensuring the RNN receives clean, scaled, and chronologically consistent inputs.

**Feature Selection (Noise Reduction)**

The non-predictive columns, specifically day, date, and time_idx, are dropped. Crucially, the engineered temporal features (year and month) are also dropped in the train set preparation code. Therefore, the final feature set for the RNN input consists only of the 5 financial metrics: open, close, low, high, and volume. This results in a consistent feature vector size of 5. This selection provides the RNN model solely with market data. 

**Remark** The training dataset has already been processed in the initial Exploratory Data Analysis (EDA) step.

In [ ]:
train = train.drop(columns=['day','date','month','year','time_idx'])
test = test.drop(columns=['date'])
train

**Data Normalization (Min-Max Scaling)**

Normalization is a mandatory step for training deep neural networks. Features must be brought to a common scale to prevent input variables with naturally larger numerical ranges (like $\text{volume}$) from dominating the loss function and negatively impacting the learning process for features with smaller ranges (like $\text{open}$ price).

Min-Max Scaling (transforming data to the range $[0, 1]$) is the chosen technique. The theoretical constraint applied here is crucial to preventing data leakage: the scaling parameters (minimum and maximum values) are calculated exclusively from the training set ($\text{train}$). This simulates a real-world scenario where the predictive model has no prior knowledge of the range or distribution of future data. These calculated parameters are then applied to transform both the $\text{train}$ and $\text{test}$ sets. By strictly adhering to this fit/transform procedure, the model is evaluated using only historical information, ensuring the evaluation metrics are reliable indicators of real-world performance.

In [ ]:
print("\n--- Applying Min-Max Scaling to Numerical Features ---")

numerical_cols = ['open', 'close', 'low', 'high', 'volume']
scaler = MinMaxScaler()

train[numerical_cols] = scaler.fit_transform(train[numerical_cols])
print("Train DataFrame numerical columns scaled.")

test[numerical_cols] = scaler.transform(test[numerical_cols])
print("Test DataFrame numerical columns transformed.")

**Sliding Window Transformation: Sequence Creation for Recurrent Networks**

Recurrent Neural Networks, including the LSTM, require input data to be structured as sequences or "time windows," not as independent, single-row observations.

The Sliding Window Technique is the standard approach for transforming a linear time series into a suitable 3D array for RNNs. The project uses a sequence length (`seq_len`) of 30 days. The concept is as follows: for any point in time $t$, the input sequence ($\mathbf{X}$) consists of the 30 days preceding the target date ($t-29$ to $t$). The corresponding target ($\mathbf{y}$) is the feature vector for the next single time step ($t+1$), specifically targeting only the 5 price columns. This process transforms the 2D feature data into the required 3D format: $(\text{Number of Samples}, \text{Sequence Length}, \text{Number of Features})$.

**Chronological Splitting of Sequences**

The sequence creation process respects the initial chronological split and also creates a validation set:

- Training Sequences: Sequences generated from the $\text{train}$ data are chronologically split, reserving $20%$ of the latest sequences for the $\mathbf{x}_{\text{valid}}$ and $\mathbf{y}_{\text{valid}}$ sets. This ensures the validation set represents the most recent historical patterns before the final test period.
- Test Sequences: A separate function generates sequences from the $\text{test}$ data. Since the test data is the dedicated future holdout, all sequences generated from it are used for the final evaluation $\mathbf{x}_{\text{test}}$ and $\mathbf{y}_{\text{test}}$.

The resulting shapes confirm the successful transformation. The input sequences ($\mathbf{x}_{\text{train}}, \mathbf{x}_{\text{valid}}, \mathbf{x}_{\text{test}}$) have the shape $\text{(N, 30, 5)}$, where 30 is the $\text{seq\_len}$ and 5 is the feature size. The single-step target predictions ($\mathbf{y}_{\text{train}}, \mathbf{y}_{\text{valid}}, \mathbf{y}_{\text{test}}$) have the shape $\text{(N, 5)}$.

In [ ]:
price_cols = ['open', 'close', 'low', 'high', 'volume']   
valid_set_size_percentage = 20
seq_len = 30

def create_sequences(df: pd.DataFrame,
                    seq_len: int,
                    valid_pct: int = 20):
    """
    Build (x_train, y_train, x_valid, y_valid) from a DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        Full training data (all columns).
    seq_len : int
        Number of past timesteps that form one input sample.
    valid_pct : int
        Percentage of *generated* sequences that go to validation.

    Returns
    -------
    tuple
        (x_train, y_train, x_valid, y_valid)
        * x_* : shape (samples, seq_len, n_features)
        * y_* : shape (samples, 5)   only the 5 price columns
    """

    data = df.values

    sequences = []
    for i in range(len(data) - seq_len):
        sequences.append(data[i : i + seq_len + 1])
    sequences = np.array(sequences, dtype=np.float32)

    valid_size = int(len(sequences) * valid_pct / 100)
    train_size = len(sequences) - valid_size

    x_train = sequences[:train_size, :-1, :]
    x_valid = sequences[train_size:, :-1, :]

    price_idx = [df.columns.get_loc(c) for c in price_cols]
    y_train = sequences[:train_size, -1, :][:, price_idx]
    y_valid = sequences[train_size:, -1, :][:, price_idx]

    return x_train, y_train, x_valid, y_valid

def create_test_sequences(df: pd.DataFrame, seq_len: int):
    data = df.values
    sequences = []
    for i in range(len(data) - seq_len):
        sequences.append(data[i : i + seq_len + 1])
    sequences = np.array(sequences, dtype=np.float32)

    price_idx = [df.columns.get_loc(c) for c in price_cols]

    x_test = sequences[:, :-1, :]
    y_test = sequences[:, -1, :][:, price_idx]
    return x_test, y_test

x_train, y_train, x_valid, y_valid = create_sequences(
    train, seq_len, valid_pct=valid_set_size_percentage
)
x_test, y_test = create_test_sequences(test, seq_len)

print('x_train.shape = ',x_train.shape)
print('y_train.shape = ', y_train.shape)
print('x_valid.shape = ',x_valid.shape)
print('y_valid.shape = ', y_valid.shape)
print('x_test.shape = ', x_test.shape)
print('y_test.shape = ',y_test.shape)

**TensorFLow Tensor Conversion**

The final NumPy arrays are converted to `tf.Tensor` objects using the `tf.constant` constructor with `tf.float32` precision. This conversion is necessary for optimal data handling and acceleration on hardware like GPUs or TPUs within the TensorFlow ecosystem.

In [ ]:
x_train_tf = tf.constant(x_train, dtype=tf.float32)
y_train_tf = tf.constant(y_train, dtype=tf.float32)
x_valid_tf = tf.constant(x_valid, dtype=tf.float32)
y_valid_tf = tf.constant(y_valid, dtype=tf.float32)
x_test_tf  = tf.constant(x_test,  dtype=tf.float32)
y_test_tf  = tf.constant(y_test,  dtype=tf.float32)

print('x_train.shape =', x_train.shape)      
print('y_train.shape =', y_train.shape)   
print('x_valid.shape =', x_valid.shape)
print('y_valid.shape =', y_valid.shape)
print('x_test.shape  =', x_test.shape)
print('y_test.shape  =', y_test.shape)

print("\nTensor shapes:")
print(f"x_train_tf {x_train_tf.shape}, y_train_tf {y_train_tf.shape}")
print(f"x_valid_tf {x_valid_tf.shape}, y_valid_tf {y_valid_tf.shape}")
print(f"x_test_tf  {x_test_tf.shape},  y_test_tf  {y_test_tf.shape}")

# Model Training

## Model Architecture and Model Building (Keras)

**RNN Architecture in Keras**

Keras simplifies the creation of sequence models by providing highly optimized, pre-built recurrent layers. The fundamental requirement for any RNN layer is a $3D$ input tensor of shape (samples, timesteps, features).

**1. Defining the Input Layer**

Before any recurrent layer can be added, the input shape must be explicitly defined using the Input layer, which sets the dimensional structure of the time series data entering the model.The required shape is $(Sequence Length, Number of Features)$ or $(timesteps, features)$. For example, a $30$-day lookback window with $5$ features would use `shape=(30, 5)`.

**2. Stabilization and Pre-Processing**

It is a best practice to stabilize the activations before they enter the recurrent part of the network, especially since RNNs are sensitive to scale.
- Layer Normalization: The `LayerNormalization(axis=-1)` layer is crucial. It normalizes the inputs across the feature dimension for each time step, independently for every sample in the batch. This standardization helps mitigate internal covariate shift, leading to more stable and faster training in deep sequence models.

**3. Stacking Recurrent Layers (Deep RNNs)**

The process of creating a deep RNN involves chaining multiple recurrent layers. The key to stacking is the `return_sequences` argument, which dictates the output shape of the layer.
- Intermediate RNN Layers: For all but the final recurrent layer, set `return_sequences=True`. This ensures the layer outputs a sequence of hidden states for every input time step, maintaining the $3D$ shape (`None, timesteps, hidden_units`). This output sequence then serves as the input sequence for the next RNN layer in the stack.
- Final RNN Layer: For the last recurrent layer before the output layer, set return_sequences=False (or omit the argument, as it defaults to False). This layer outputs only the final hidden state—a $2D$ vector of shape (None, hidden_units)—which is a compressed summary of the entire input sequence.

**4. Regularization and Initialization**

To combat the inherent tendency of RNNs to overfit or suffer from unstable gradients, Dropout and specific initializers are applied to each recurrent layer.
- Dropout: The Dropout layer is essential and should be placed after each recurrent layer. It randomly drops a percentage of the connections, which regularizes the network and improves generalization. Dropout can also be applied inside the recurrent layer by setting the dropout argument (often preferred for recurrent connections) and the `recurrent_dropout` argument (often set to $0$ in Keras due to implementation limitations).
- Initializers: Specifying initializers helps set the initial weight magnitudes. For example, `glorot_uniform` is typically used for the kernel (input-to-hidden weights), and orthogonal is often used for the recurrent weights (hidden-to-hidden weights), which assists in maintaining stable gradients.

**5. The Output Layer**

The final output vector from the recurrent stack (the $2D$ summary vector) is passed to a standard Dense layer.The number of units in this layer corresponds exactly to the number of output features being predicted (e.g., $5$ units for $5$ price/volume predictions). This layer linearly transforms the $2D$ sequence summary into the final prediction vector.

**RNN implementation using Keras**

This section details the construction and execution of a three-layer SimpleRNN model in Keras, emphasizing the theoretical components, necessary regularization, and the optimization strategy used for time series forecasting.

**Model Architecture and Regularization**

The forecasting model is constructed as a Stacked SimpleRNN architecture, where multiple recurrent layers are chained to increase the model's capacity to learn complex temporal dependencies.
- Input Layer: The model receives $30$ time steps, each containing $5$ features (open, close, low, high, volume). The input shape is $(30, 5)$.
- Layer Normalization: A LayerNormalization layer is placed immediately after the input. This technique normalizes the activations across the feature dimension (`axis=-1`) for a single time step and batch sample. Unlike Batch Normalization, Layer Normalization's statistics are calculated independently for each sample, making it highly effective for sequence models where batch sizes can be small and input sequences are variable. It standardizes the inputs to the recurrent layers, which significantly aids in stabilizing training and ensuring faster convergence.
- Stacked SimpleRNN Layers: Three SimpleRNN layers, each with $128$ hidden units, form the core of the model.
    - The first two RNN layers use `return_sequences=True` to pass the entire sequence of $30$ hidden states to the next layer. This enables deep sequence processing.
    - The final RNN layer uses `return_sequences=False`, which outputs only the final hidden state—a single vector of $128$ features that summarizes the $30$-day input sequence for the final prediction.
- Dropout: A Dropout layer with a rate of $0.25$ is applied after each recurrent layer. Dropout randomly sets a fraction of input units to zero during training. In RNNs, this acts as a form of regularization, preventing the model from relying too heavily on specific hidden states and dramatically reducing overfitting.
- Weight Initialization: `glorot_uniform` (Xavier) is used for the kernel weights, and orthogonal initialization is used for the recurrent weights. Orthogonal initialization helps maintain the scale of gradients during backpropagation through time, slightly mitigating the vanishing/exploding gradient problems inherent to RNNs.
- Output Layer: A final Dense layer with $5$ units maps the $128$-dimensional summary vector to the $5$ target features (the next day's $5$ prices/volume).

**Compilation and Training Strategy**

The model is optimized using a robust setup designed to maximize performance while ensuring generalization to the unseen validation data.
- Optimizer and Gradient Clipping: The Adam optimizer is chosen for its efficiency in handling sparse gradients. A crucial addition is Gradient Clipping (clipnorm=1.0), which limits the maximum magnitude of the gradients during backpropagation. In RNNs, the recursive nature of the gradient computation over many time steps can lead to exploding gradients; clipping prevents these runaway updates, ensuring stability.
- Loss Function: Mean Squared Error (MSE) is used, as this is a standard and differentiable loss function for regression tasks, measuring the average squared difference between the predicted and actual scaled values.
- Early Stopping: An EarlyStopping callback monitors the `val_loss` with a patience of $20$ epochs. This is the primary mechanism for fighting overfitting, as training is halted once the validation error ceases to improve, and the weights from the best performing epoch (epoch 11 in this case, where `val_loss` was $0.0181$) are restored.
- Learning Rate Scheduling: ReduceLROnPlateau is used to dynamically adjust the learning rate during training. If the `val_loss` does not improve for a specified number of epochs (patience of $8$), the learning rate is automatically reduced by a factor of $0.5$. This allows the model to jump out of early, high-learning-rate plateaus and search for finer minima in the loss landscape.

In [ ]:
seq_len    = x_train_tf.shape[1]
input_size = x_train_tf.shape[2]
hidden     = 128
dropout    = 0.25
output_sz  = y_train_tf.shape[1]

model = keras.Sequential([
    layers.Input(shape=(seq_len, input_size)),

    layers.LayerNormalization(axis=-1),

    layers.SimpleRNN(hidden, return_sequences=True,
                     dropout=dropout,
                     kernel_initializer='glorot_uniform',
                     recurrent_initializer='orthogonal'),

    layers.Dropout(dropout),

    layers.SimpleRNN(hidden, return_sequences=True,
                     dropout=dropout,
                     kernel_initializer='glorot_uniform',
                     recurrent_initializer='orthogonal'),

    layers.Dropout(dropout),

    layers.SimpleRNN(hidden, return_sequences=False,
                     dropout=dropout,
                     kernel_initializer='glorot_uniform',
                     recurrent_initializer='orthogonal'),

    layers.Dropout(dropout),

    layers.Dense(output_sz)
])

model.summary()
print(f"Trainable params: {model.count_params():,}")

## Model Training (Keras)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
    loss=keras.losses.MeanSquaredError()
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

lr_reduce = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=8,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    x_train_tf, y_train_tf,
    validation_data=(x_valid_tf, y_valid_tf),
    epochs=300,
    batch_size=64,
    callbacks=[early_stop, lr_reduce],
    verbose=1
)

print("\nTraining finished.")

**Interpretation of Training History**

The training output reveals the effects of the defined architecture and callbacks:
- Rapid Initial Learning: The loss drops quickly across the first few epochs (from $0.8047$ to $0.0912$), indicating the model quickly learned the general trend and dependencies in the training data.
- Validation Performance: The validation loss (`val_loss`) initially decreases, reaching a minimum at Epoch 11 ($0.0181$).
- Overfitting Detection: After Epoch 11, the training loss continues to decrease, but the validation loss begins to increase and fluctuate widely. This confirms that the model starts to overfit to the training data.
- Callback Activation: The ReduceLROnPlateau callback is triggered first (at Epoch 19 and again at Epoch 27) as the validation loss stalls. The EarlyStopping callback subsequently activates when no further improvement in `val_loss` is recorded within the patience window, terminating training early at Epoch 31 and restoring the best weights from Epoch 11. This process ensures the model retains the weights that yielded the highest generalization performance.

**Training History (Keras)**

The training history plot tracks the Mean Squared Error (MSE), which serves as the loss function, across epochs for both the training and validation datasets.
- Training Loss (Blue Line): This measures the error on the data the model sees during optimization. A continuously decreasing training loss indicates the model is successfully learning the patterns within the training set.
- Validation Loss (Orange Line): This measures the model's error on the sequestered validation data. It is the primary indicator of generalization performance.
- Overfitting: The critical point in the plot occurs when the training loss continues to decrease while the validation loss plateaus or begins to increase. This gap signifies overfitting, meaning the model has started memorizing noise or specific patterns unique to the training data rather than learning general rules. The Early Stopping callback is designed to halt training just before this point becomes severe, restoring the best weights (which occurred at Epoch 11, where the validation loss was lowest).

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (Mean Squared Error)')
plt.legend()
plt.grid(True)
plt.show()

**Prediction Results Visualization (Validation Dataset)**

The predictions, initially calculated on the normalized data scale, must be converted back to the original price space for meaningful interpretation.
- Inference: The model is used to generate predictions (`model.predict()`) on the training and validation sets in a deterministic, non-training manner.
- Inverse Transformation: The predictions are scaled back using the MinMaxScaler object. Since the scaler was fit only on the original training features, applying its `inverse_transform` ensures that the predicted values are returned to their original dollar-price and volume ranges. This step is essential for calculating real-world error metrics like RMSE.

In [ ]:
y_train_pred = model.predict(x_train_tf)
y_valid_pred = model.predict(x_valid_tf)

y_train_pred_orig = scaler.inverse_transform(y_train_pred)
y_valid_pred_orig = scaler.inverse_transform(y_valid_pred)
y_train_actual_orig = scaler.inverse_transform(y_train_tf.numpy())
y_valid_actual_orig = scaler.inverse_transform(y_valid_tf.numpy())

ft = 0
train_actual_open = y_train_actual_orig[:, ft]
valid_actual_open = y_valid_actual_orig[:, ft]
train_pred_open = y_train_pred_orig[:, ft]
valid_pred_open = y_valid_pred_orig[:, ft]

train_start = seq_len
valid_start = seq_len + len(train_actual_open)

time_train = np.arange(train_start, train_start + len(train_actual_open))
time_valid = np.arange(valid_start, valid_start + len(valid_actual_open))

In [ ]:
plt.figure(figsize=(16, 8))

plt.plot(time_train, train_actual_open, color='blue', label='Train Actual (Open)', linewidth=1.6)
plt.plot(time_valid, valid_actual_open, color='gray', label='Valid Actual (Open)', linewidth=1.6)
plt.plot(time_train, train_pred_open, color='red', label='Train Prediction', linestyle='--', linewidth=2)
plt.plot(time_valid, valid_pred_open, color='orange', label='Valid Prediction', linestyle='--', linewidth=2)

plt.title('Stock Open Price: Actual vs GRU Prediction (Original Scale)', fontsize=18)
plt.xlabel('Time [Days]', fontsize=14)
plt.ylabel('Price ($)', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)

train_rmse = np.sqrt(np.mean((train_actual_open - train_pred_open) ** 2))
valid_rmse = np.sqrt(np.mean((valid_actual_open - valid_pred_open) ** 2))
plt.tight_layout()
plt.show()

print(f'Train RMSE: {train_rmse:,.2f}')
print(f'\nValid RMSE: {valid_rmse:,.2f}')

The full-timeline plot and calculated metrics provide a quantitative and qualitative assessment of the model's success.

**Interpretation of Prediction Visualization**

The visualization displays the actual open prices against the predicted open prices over the complete historical period (train and valid).
- Training Fit (Blue vs. Red Dashed Line): The training prediction (red dashed line) tracks the actual training price (blue line) very closely. This demonstrates that the Recurrent Neural Network (RNN) has successfully utilized its high capacity to fit the complex, non-linear dependencies in the historical data, resulting in a low training error.
- Validation Generalization (Gray vs. Orange Dashed Line): On the validation set, the prediction (orange dashed line) shows a noticeable increase in lag and a smoother trajectory compared to the actual price. While the model generally follows the macro trend of the stock price, it struggles to capture the day-to-day volatility and local turning points. This behavior is consistent with the model exhibiting generalization error and confirms the early signs of overfitting identified in the loss plot.

**Interpretation of Quantitative Metrics**

Two key metrics are calculated on the original price scale:
- Root Mean Squared Error (RMSE): RMSE measures the average magnitude of the prediction error in the original units (dollars).
- Train RMSE ($50.47$): This confirms a strong fit to the training data.
- Valid RMSE ($29.35$): This dollar-error is lower than the training RMSE, which is unusual for a perfect fit, but likely indicates the validation subset had less internal variance or volatility than the training subset. Crucially, this value represents the average dollar-miss on unseen data.

In [ ]:
def sign_accuracy(y_actual, y_pred):
    actual_diff = y_actual[:, 1] - y_actual[:, 0]  # close - open
    pred_diff = y_pred[:, 1] - y_pred[:, 0]
    return np.mean(np.sign(actual_diff) == np.sign(pred_diff))

train_sign = sign_accuracy(y_train_actual_orig, y_train_pred_orig)
valid_sign = sign_accuracy(y_valid_actual_orig, y_valid_pred_orig)

print(f"\nDirection Accuracy (Close > Open):")
print(f"  Train: {train_sign:.3f} ({train_sign*100:.1f}%)")
print(f"  Valid: {valid_sign:.3f} ({valid_sign*100:.1f}%)")

Directional Accuracy (Sign Accuracy): This metric assesses the model's ability to predict whether the price will go up or down, specifically comparing the predicted change from open to close against the actual change.
- Train Directional Accuracy ($50.1%$): A figure very close to $50%$ suggests the model has not learned a meaningful pattern for predicting the daily direction on the training set.
- Valid Directional Accuracy ($47.5%$): Falling below $50%$ on the validation set confirms that the model's directional forecasts are worse than random chance. This is a major limitation, indicating that while the RNN can track price magnitudes, it fails at the finer, more complex task of forecasting short-term directional movement.

## Model Architecture and Model Building (TensorFlow custom functions)

**RNN using the Keras Custom Functions**

Creating a custom layer is the most common and robust way to implement novel architecture blocks within the Keras ecosystem.

**1. Defining the Custom Layer Class (`tf.keras.layers.Layer`)**

The custom layer inherits from `tf.keras.layers.Layer` and typically overrides three key methods:
- `__init__(self, ...)`: This is where the internal, non-trainable components of the layer are initialized. For an RNN block, this involves declaring necessary layers like LayerNormalization and individual SimpleRNN (or $\text{LSTM}$/$\text{GRU}$) instances. This ensures these sub-layers are created only once and track their weights correctly.
- `build(self, input_shape)`: Optional but recommended. This method is called once with the known input shape to create the layer's trainable weights using `self.add_weight()`. For an RNN block that utilizes existing Keras RNN cells, this method is often skipped, as the sub-layers handle weight creation implicitly.
- `call(self, inputs)`: This defines the layer's forward pass logic, detailing how the input tensor is processed through the defined sub-layers. For a stacked $\text{RNN}$ block, this involves applying normalization, iterating through the list of recurrent layers, and passing the output of one to the input of the next. The core sequence logic (`return_sequences=True` for intermediate layers, False for the final layer) is implemented here.

**2. Implementing Custom Logic within the call Method**

The call method orchestrates the data flow, which is where high-level architectural decisions are implemented:
- Input Preprocessing: Apply stabilization techniques, such as Layer Normalization, to the input sequence tensor before it enters the first recurrent unit.
- Sequential Stacking: Utilize a loop to process the input through each recurrent layer in sequence. This loop must ensure that all but the final recurrent layer output a $3D$ sequence tensor, effectively creating a deep, stacked $\text{RNN}$.
- Output Dimensionality: The last recurrent layer processes the sequence and returns a $2D$ summary vector, which becomes the output of the custom block for downstream layers like Dense or BatchNormalization.

**Approach using TensorFlow Custom Functions (`tf.function`) for Optimization**

While the Keras Custom Layer defines the structure, `tf.function` is used to optimize the execution speed of Python code by converting it into a static, high-performance TensorFlow graph.

**Integrating `tf.function`**

For most Keras custom layers, `tf.function` is not strictly necessary on the call method because Keras already handles graph construction. However, when developing a completely custom RNN cell from scratch (defining the recurrent step computation manually using `tf.scan or tf.while_loop`), `tf.function` becomes vital:
- Custom Cell Implementation: If one were to write a completely new recurrent cell (e.g., a variant of $\text{GRU}$ or $\text{LSTM}$), the recurrent step function itself would be wrapped in `@tf.function`. This decorator traces the Python logic to generate an optimized, single TensorFlow graph representing the unrolled $\text{RNN}$ computation, which significantly speeds up both training and inference.
- Performance Benefits: By converting the eager execution operations (standard Python code) into a graph, `tf.function` enables compiler optimizations such as function inlining and static shape inference. This eliminates Python overhead and improves utilization of underlying hardware (GPUs/TPUs).

**Conclusion on Integration**

The standard pattern is to use the Keras Custom Layer to define the RNN's structure and trainable parameters. If that layer relies on complex, manually defined TensorFlow operations (like a custom activation function or a non-standard weight update logic), those specific functions should be wrapped with `@tf.function` to ensure optimized graph execution within the custom layer's call method.

**RNN implementation using TensorFlow**

This model employs a sophisticated approach to deep Recurrent Neural Networks (RNNs) by encapsulating the recurrent logic within a custom Keras layer, providing modularity and precise control over the stacking and normalization processes.

**1. Custom Layer Design: CustomRNN**

BlockThe core architectural enhancement is the use of a Keras layers.Layer subclass, CustomRNNBlock, which handles both the preprocessing and the deep stacking of the SimpleRNN cells.
- Layer Encapsulation: This custom block initializes and manages multiple SimpleRNN layers internally, determined by the `num_rnn_layers` parameter (set to $2$ in this implementation).
- Pre-Recurrent Normalization: A LayerNormalization layer is placed at the beginning of the call method. This ensures that the input sequence features are normalized immediately prior to entering the first recurrent layer, which is essential for stabilizing gradient flow across the sequence dimension and preventing early saturation of the recurrent unit activations.
- Controlled Stacking: The logic inside the custom block automatically manages the return_sequences argument for each internal SimpleRNN layer. All intermediate layers are set to return_sequences=True (outputting the full $3D$ sequence), while the final layer is set to return_sequences=False (outputting only the $2D$ summary vector). This ensures the recurrent block functions correctly as a sequence-to-vector encoder.
- Parameter Efficiency: By wrapping the layers, the model's structure in the main sequential API becomes cleaner and more abstract. The entire block contributes $50,058$ trainable parameters (the sum of weights and biases from the two stacked RNNs and the Layer Normalization).

2. Sequential Model Construction

The main model leverages the CustomRNNBlock as a single, powerful component.
- Input and Custom Block: The model starts with the Input layer, defining the $(30, 5)$ sequence shape, followed immediately by the $2$-layer CustomRNNBlock with $128$ hidden units and $0.2$ dropout.
- Post-Block Normalization: Following the custom block, a BatchNormalization layer is introduced. This is unusual but serves to stabilize the final $2D$ feature vector ($128$ features) produced by the RNN stack before the prediction. While Layer Normalization handles the input sequence, Batch Normalization here helps standardize the activations across the batch dimension for the final features.
- Regularization and Output: A standard Dropout layer further regularizes the final $128$-feature vector. The output is then mapped to the $5$ target features by the Dense layer.

**3. Compilation and Training Strategy**

The training process uses standardized techniques necessary for deep sequence learning.
- Optimizer and Clipping: The Adam optimizer with a base learning rate of $1e-3$ is used. Crucially, gradient clipping (clipnorm=1.0) is enforced. This bounds the magnitude of the gradients, preventing the exploding gradient problem, which is common in deep or long sequence RNNs due to repeated multiplication of weights during backpropagation through time.
- Loss Function: Mean Squared Error (MSE) is the objective function, appropriate for a regression task predicting continuous stock prices and volumes.Callbacks for Stability:
- Early Stopping: This monitors val_loss with a patient setting of $18$ epochs. This mechanism conserves training resources and directly combats overfitting by ensuring the model stops training when generalization performance ceases to improve, ultimately restoring the weights from the best epoch.
- ReduceLROnPlateau: This adaptive learning rate scheduler reduces the learning rate by half (factor $0.5$) if the validation loss plateaus for $6$ epochs. This allows the optimizer to escape local minima and refine the model's weights in flatter regions of the loss landscape.

In [ ]:
# CUSTOM RNN BLOCK
class CustomRNNBlock(layers.Layer):
    def __init__(self, hidden_size, num_rnn_layers, dropout_rate, **kwargs):
        super().__init__(**kwargs)
        self.hidden_size = hidden_size
        self.num_rnn_layers = num_rnn_layers
        self.dropout_rate = dropout_rate

        self.norm = layers.LayerNormalization(axis=-1)
        self.rnn_layers = [
            layers.SimpleRNN(
                hidden_size,
                return_sequences=(i < num_rnn_layers - 1),
                dropout=dropout_rate,
                kernel_initializer='glorot_uniform',
                recurrent_initializer='orthogonal'
            )
            for i in range(num_rnn_layers)
        ]

    def call(self, inputs):
        x = self.norm(inputs)
        for rnn in self.rnn_layers:
            x = rnn(x)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            'hidden_size': self.hidden_size,
            'num_rnn_layers': self.num_rnn_layers,
            'dropout_rate': self.dropout_rate,
        })
        return config

# MODEL PARAMETERS
seq_len     = x_train_tf.shape[1]      # 30
input_size  = x_train_tf.shape[2]      # 5
hidden_size = 128
num_layers  = 2          # only 2 RNN layers
output_size = y_train_tf.shape[1]   # 5
dropout_rate = 0.2

# BUILD MODEL (CustomRNNBlock + BatchNorm + Dense)
model = keras.Sequential([
    layers.Input(shape=(seq_len, input_size)),
    CustomRNNBlock(hidden_size=hidden_size,
                   num_rnn_layers=num_layers,
                   dropout_rate=dropout_rate),
    layers.BatchNormalization(),        # stabilises custom block
    layers.Dropout(dropout_rate),
    layers.Dense(output_size)
])

model.summary()

## Model Training (TensorFlow custom functions)

In [ ]:
# COMPILE (sane LR + gradient clipping)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
    loss=keras.losses.MeanSquaredError()
)

# CALLBACKS
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=18,               
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=6,
    min_lr=1e-6,
    verbose=1
)

# TRAIN
history = model.fit(
    x_train_tf, y_train_tf,
    validation_data=(x_valid_tf, y_valid_tf),
    epochs=300,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

**Training History (TensorFlow custom functions)**

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (Mean Squared Error)')
plt.legend()
plt.grid(True)
plt.show()

**Prediction Results Visualization (Validation Dataset)**

In [ ]:
y_train_pred = model.predict(x_train_tf)
y_valid_pred = model.predict(x_valid_tf)

y_train_pred_orig = scaler.inverse_transform(y_train_pred)
y_valid_pred_orig = scaler.inverse_transform(y_valid_pred)
y_train_actual_orig = scaler.inverse_transform(y_train_tf.numpy())
y_valid_actual_orig = scaler.inverse_transform(y_valid_tf.numpy())

ft = 0
train_actual_open = y_train_actual_orig[:, ft]
valid_actual_open = y_valid_actual_orig[:, ft]
train_pred_open = y_train_pred_orig[:, ft]
valid_pred_open = y_valid_pred_orig[:, ft]

train_start = seq_len
valid_start = seq_len + len(train_actual_open)

time_train = np.arange(train_start, train_start + len(train_actual_open))
time_valid = np.arange(valid_start, valid_start + len(valid_actual_open))

In [ ]:
# PLOT: Actual vs Predicted (Open Price)
plt.figure(figsize=(16, 8))

plt.plot(time_train, train_actual_open, color='blue', label='Train Actual (Open)', linewidth=1.6)
plt.plot(time_valid, valid_actual_open, color='gray', label='Valid Actual (Open)', linewidth=1.6)
plt.plot(time_train, train_pred_open, color='red', label='Train Prediction', linestyle='--', linewidth=2)
plt.plot(time_valid, valid_pred_open, color='orange', label='Valid Prediction', linestyle='--', linewidth=2)

plt.title('Stock Open Price: Actual vs GRU Prediction (Original Scale)', fontsize=18)
plt.xlabel('Time [Days]', fontsize=14)
plt.ylabel('Price ($)', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)

# Metrics
train_rmse = np.sqrt(np.mean((train_actual_open - train_pred_open) ** 2))
valid_rmse = np.sqrt(np.mean((valid_actual_open - valid_pred_open) ** 2))
plt.tight_layout()
plt.show()

print(f'Train RMSE: {train_rmse:,.2f}')
print(f'\nValid RMSE: {valid_rmse:,.2f}')

**Interpretation of Prediction Visualization**

The full timeline plot compares the model's forecasts against the actual historical prices, offering a qualitative assessment alongside quantitative error metrics. The plot shows the actual Open Price against the model's predicted Open Price over both the training and validation periods.

- Training Fit (Blue vs. Red Dashed Line): The training prediction (red dashed line) adheres very closely to the actual training price (blue line). This strong fit indicates that the deep Simple RNN architecture possesses sufficient complexity to model the long-term dependencies and non-linear patterns within the seen data, resulting in a low training error.
- Validation Generalization (Gray vs. Orange Dashed Line): On the validation set, the prediction (orange dashed line) maintains a respectable fidelity to the overall price trajectory. It successfully captures the major trends, demonstrating its capacity for generalization. However, the prediction lags the sharpest price fluctuations, indicating that the model smooths out short-term volatility, a common characteristic of recurrent models.

**Quantitative Metrics Validation**

In [ ]:
# SIGN ACCURACY (Close - Open Direction)
def sign_accuracy(y_actual, y_pred):
    actual_diff = y_actual[:, 1] - y_actual[:, 0]  # close - open
    pred_diff = y_pred[:, 1] - y_pred[:, 0]
    return np.mean(np.sign(actual_diff) == np.sign(pred_diff))

train_sign = sign_accuracy(y_train_actual_orig, y_train_pred_orig)
valid_sign = sign_accuracy(y_valid_actual_orig, y_valid_pred_orig)

print(f"\nDirection Accuracy (Close > Open):")
print(f"  Train: {train_sign:.3f} ({train_sign*100:.1f}%)")
print(f"  Valid: {valid_sign:.3f} ({valid_sign*100:.1f}%)")

**Interpretation of Quantitative Metrics**

- Root Mean Squared Error (RMSE): RMSE measures the average magnitude of the prediction error in the original price units.
    - Train RMSE ($21.94$): A very low value confirming the high fidelity of the fit to the training data.
    - Valid RMSE ($44.19$): This value is nearly double the training RMSE, which is a clear indicator of the bias-variance tradeoff and the model's difficulty generalizing to unseen volatility compared to its performance on the training set. This margin highlights the degree of overfitting (as managed by early stopping).

- Directional Accuracy (Sign Accuracy): This metric assesses the model's ability to forecast the relative change in price (Close $>$ Open).
    - Train Directional Accuracy ($53.4%$): This result is only slightly better than a random coin flip ($50%$), suggesting the model has limited ability to learn the subtle factors governing daily price movement direction, even on the training data.
    - Valid Directional Accuracy ($51.8%$): The accuracy remains near chance level on the unseen data. This confirms a significant limitation: the RNN is proficient at tracking the long-term magnitude of the price series but struggles with the stochastic, short-term decision of directional change, a far more challenging task in financial forecasting.

# Model Evaluation

This final stage quantifies the model's true performance using the completely unseen test dataset and visualizes its predictive capabilities across all three data splits (train, validation, and test).

**Regression Evaluation (MSE)**

In [ ]:
print("\n--- Final Test Evaluation ---")

final_test_loss = model.evaluate(x_test_tf, y_test_tf, verbose=0)
print(f"Final Test Loss (MSE): {final_test_loss:.6f}")

**Test Loss and Generalization Theory**

Model evaluation begins with calculating the final loss on the test set, which serves as the most objective measure of the model's ability to generalize to new data.
- Final Test Loss (MSE: $0.196569$): The Mean Squared Error (MSE) on the scaled test set is significantly higher than the training loss (which stabilized much lower) and substantially higher than the best validation loss ($0.0225$).
- Generalization Gap: This gap between the validation/training loss and the test loss indicates that the model's performance degraded considerably on the final, un-interacted-with data. This is often the ultimate consequence of the bias-variance tradeoff; despite aggressive regularization (dropout, early stopping), the model overfit to the specific characteristics of the training and validation periods and struggled with the potentially different statistical distribution or volatility present in the test period.

**Prediction Results Visualization (Test Dataset)**

In [ ]:
y_train_pred = model.predict(x_train_tf)
y_valid_pred = model.predict(x_valid_tf)
y_test_pred  = model.predict(x_test_tf)

y_train_actual = y_train_tf.numpy()
y_valid_actual = y_valid_tf.numpy()
y_test_actual  = y_test_tf.numpy()

y_train_pred_orig = scaler.inverse_transform(y_train_pred)
y_valid_pred_orig = scaler.inverse_transform(y_valid_pred)
y_test_pred_orig  = scaler.inverse_transform(y_test_pred)

seq_len   = 30
day_start = seq_len

n_train = len(y_train_actual)
n_valid = len(y_valid_actual)
n_test  = len(y_test_actual)

time_train = np.arange(day_start, day_start + n_train)
time_valid = np.arange(day_start + n_train,
                       day_start + n_train + n_valid)
time_test  = np.arange(day_start + n_train + n_valid,
                       day_start + n_train + n_valid + n_test)

In [ ]:
ft = 0                                 # 0 = open price
plt.figure(figsize=(18, 6))

# LEFT: Full history
ax1 = plt.subplot(1, 2, 1)
ax1.plot(time_train, scaler.inverse_transform(y_train_actual)[:, ft],
         color='blue',   label='Train Target')
ax1.plot(time_valid, scaler.inverse_transform(y_valid_actual)[:, ft],
         color='gray',   label='Valid Target')
ax1.plot(time_test,  scaler.inverse_transform(y_test_actual)[:,  ft],
         color='black',  label='Test Target')

ax1.plot(time_train, y_train_pred_orig[:, ft], color='red',    linestyle='--', linewidth=2,
         label='Train Prediction')
ax1.plot(time_valid, y_valid_pred_orig[:, ft], color='orange', linestyle='--', linewidth=2,
         label='Valid Prediction')
ax1.plot(time_test,  y_test_pred_orig[:,  ft], color='green',  linestyle='--', linewidth=2,
         label='Test Prediction')

ax1.set_title('Past and Future Stock Prices (Open Price)', fontsize=14)
ax1.set_xlabel('Time [days]')
ax1.set_ylabel('Price ($)')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# RIGHT: Test zoom
ax2 = plt.subplot(1, 2, 2)
ax2.plot(time_test, scaler.inverse_transform(y_test_actual)[:, ft],
         color='black', label='Test Target')
ax2.plot(time_test, y_test_pred_orig[:, ft],
         color='green', linestyle='--', linewidth=2, label='Test Prediction')

ax2.set_title('Future Stock Prices (Open Price)', fontsize=14)
ax2.set_xlabel('Time [days]')
ax2.set_ylabel('Price ($)')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Prediction Visualization**

The two plots provide a clear visual narrative of the model's performance over the entire time history.

- Full History Plot (Left): The visualization confirms that the model's Train Prediction (red dashed line) adheres closely to the Train Target (blue), demonstrating high capacity and successful parameter optimization. However, the divergence increases sequentially: the Valid Prediction (orange dashed line) shows more smoothing and lag, and the Test Prediction (green dashed line) exhibits the greatest divergence, particularly struggling to capture the magnitude and timing of turning points.
- Test Zoom Plot (Right): Focusing on the test period, the Test Prediction (green dashed line) clearly demonstrates excessive smoothing. It follows the overall trend but lacks the responsiveness required for accurate day-ahead forecasting. The model acts as a highly filtered tracker of the price's general direction rather than a precise predictor of specific daily values.

**Qualitative Metrics Evaluation**

In [ ]:
# RMSE (original scale)
train_rmse = np.sqrt(np.mean((scaler.inverse_transform(y_train_actual)[:, ft] -
                              y_train_pred_orig[:, ft])**2))
valid_rmse = np.sqrt(np.mean((scaler.inverse_transform(y_valid_actual)[:, ft] -
                              y_valid_pred_orig[:, ft])**2))
test_rmse  = np.sqrt(np.mean((scaler.inverse_transform(y_test_actual)[:,  ft] -
                              y_test_pred_orig[:,  ft])**2))

print("\nRMSE (Open Price):")
print(f"  Train: {train_rmse:,.2f}")
print(f"  Valid: {valid_rmse:,.2f}")
print(f"  Test:  {test_rmse:,.2f}")

# SIGN ACCURACY (Close > Open) – original scale
def sign_accuracy(y_actual, y_pred):
    actual_diff = y_actual[:, 1] - y_actual[:, 0]   # close - open
    pred_diff   = y_pred[:,   1] - y_pred[:,   0]
    return np.mean(np.sign(actual_diff) == np.sign(pred_diff))

train_sign = sign_accuracy(scaler.inverse_transform(y_train_actual),
                           y_train_pred_orig)
valid_sign = sign_accuracy(scaler.inverse_transform(y_valid_actual),
                           y_valid_pred_orig)
test_sign  = sign_accuracy(scaler.inverse_transform(y_test_actual),
                           y_test_pred_orig)

print("\nDirection Accuracy (Close > Open):")
print(f"  Train: {train_sign:.3f} ({train_sign*100:5.1f}%)")
print(f"  Valid: {valid_sign:.3f} ({valid_sign*100:5.1f}%)")
print(f"  Test:  {test_sign:.3f}  ({test_sign*100:5.1f}%)")

**Quantitative Performance Metrics**

The metrics calculated on the original dollar scale provide concrete evidence of the prediction errors.
- Root Mean Squared Error (RMSE): The Test RMSE of $126.76$ is approximately three times higher than the validation RMSE. This quantitative jump confirms the visual observation that the model's magnitude error exploded when confronted with the test data. This significant increase in error magnitude confirms poor generalization performance.
- Directional Accuracy (Sign Accuracy): The directional accuracy remains consistently close to a $50%$ baseline across all sets (Train $53.4%$, Test $52.6%$). This result implies that the recurrent model is effectively guessing the direction of the next day's price movement. While the model's continuous predictions (price magnitudes) are far from random, its ability to capture the specific, noisy drivers of daily directional change is minimal, illustrating the profound difficulty of predicting short-term stock market price movements.

# Conclusion

This structured approach provides a comprehensive methodology for building, training, and evaluating RNN models for complex time series forecasting. The process underscores the importance of a disciplined workflow, from managing chronological data splits and scaling to implementing necessary architectural and optimization techniques. The focus on Early Stopping, Gradient Clipping, and Layer Normalization is paramount for stabilizing the SimpleRNN architecture. The final evaluation, which includes both RMSE and Directional Accuracy on the unobserved test set, offers a transparent measure of the model's predictive utility.

# References

- [Recurrent neural network](https://en.wikipedia.org/wiki/Recurrent_neural_network)
- [Introduction to Recurrent Neural Networks](https://www.geeksforgeeks.org/machine-learning/introduction-to-recurrent-neural-network/)
- [Recurrent Neural Networks cheatsheet](https://stanford.edu/~shervine/teaching/cs-230/cheatsheet-recurrent-neural-networks)
- [Recurrent Neural Network (RNN) Architecture Explained](https://medium.com/@poudelsushmita878/recurrent-neural-network-rnn-architecture-explained-1d69560541ef)
- [Recurrent layers Keras](https://keras.io/api/layers/recurrent_layers/)
- [Working with RNNs TensorFlow](https://www.tensorflow.org/guide/keras/working_with_rnns)

- [DPO Trainer Hugging Face](https://huggingface.co/docs/trl/en/dpo_trainer)
- [Direct Preference Optimization: Your Language Model is Secretly a Reward Model Arxiv](https://arxiv.org/abs/2305.18290)
- [ORPO: Monolithic Preference Optimization without Reference Model Arxiv](https://arxiv.org/abs/2403.07691)
- [ORPO Trainer Hugging Face](https://huggingface.co/docs/trl/en/orpo_trainer)
- [KTO: Model Alignment as Prospect Theoretic Optimization Arxiv](https://arxiv.org/abs/2402.01306)
- [KTO Trainer Hugging Face](https://huggingface.co/docs/trl/kto_trainer)
- [Contrastive Preference Optimization: Pushing the Boundaries of LLM Performance in Machine Translation Arxiv](https://arxiv.org/abs/2401.08417)
- [CPO Trainer](https://huggingface.co/docs/trl/v0.8.5/cpo_trainer)
- [IPO: Your Language Model is Secretly a Preference Classifier Arxiv](https://arxiv.org/abs/2502.16182)
- [A General Theoretical Paradigm to Understand Learning from Human Preferences Arxiv](https://arxiv.org/abs/2310.12036)
- [Preference Tuning LLMs with Direct Preference Optimization Methods Hugging Face](https://huggingface.co/blog/pref-tuning)
- [Statistical Rejection Sampling Improves Preference Optimization](https://arxiv.org/abs/2309.06657)
- [R-PRM: Reasoning-Driven Process Reward Modeling Arxiv](https://arxiv.org/abs/2503.21295)
- [PRM Trainer Hugging Face](https://huggingface.co/docs/trl/prm_trainer)
- [Bradley-Terry and Multi-Objective Reward Modeling Are Complementary Arxiv](https://arxiv.org/abs/2507.07375)
- [Rethinking Bradley-Terry Models in Preference-Based Reward Modeling: Foundations, Theory, and Alternatives Arxiv](https://arxiv.org/abs/2411.04991)
- [Principled Reinforcement Learning with Human Feedback from Pairwise or K-wise Comparisons Arxiv](https://arxiv.org/abs/2301.11270)
- [Reward Model Ensembles Help Mitigate Overoptimization Arxiv](https://arxiv.org/abs/2310.02743)
- [Scalable Ensembling For Mitigating Reward Overoptimisation Arxiv](https://arxiv.org/abs/2406.01013)
- [Robust Preference Optimization through Reward Model Distillation Arxiv](https://arxiv.org/abs/2405.19316)
- [Offline vs. Online Reinforcement Learning Hugging Face](https://huggingface.co/learn/deep-rl-course/unitbonus3/offline-online)
- [Online and Offline Reinforcement Learning Geeksforgeeks](https://www.geeksforgeeks.org/deep-learning/online-and-offline-reinforcement-learning/)
- [Part 1: Key Concepts in RL Spinningup](https://spinningup.openai.com/en/latest/spinningup/rl_intro.html)
- [Safe Continual Reinforcement Learning Methods for Nonstationary Environments. Towards a Survey of the State of the Art Arxiv](https://arxiv.org/abs/2601.05152)
- [Revisiting Group Relative Policy Optimization: Insights into On-Policy and Off-Policy Training Arxiv](https://arxiv.org/html/2505.22257v2)
- [Group Relative Policy Optimization (GRPO) Trainer Easydel](https://easydel.readthedocs.io/en/main/trainers/grpo.html)
- [RLOO Trainer Hugging Face](https://huggingface.co/docs/trl/en/rloo_trainer)
- [REINFORCE Leave-One-Out (RLOO) Swift Readthedocs](https://swift.readthedocs.io/en/latest/Instruction/GRPO/AdvancedResearch/RLOO.html)
- [Back to Basics: Revisiting REINFORCE Style Optimization for Learning from Human Feedback in LLMs Arxiv](https://arxiv.org/html/2402.14740v1)
- [Online DPO Trainer Hugging Face](https://huggingface.co/docs/trl/online_dpo_trainer)
- [Online DPO: Online Direct Preference Optimization with Fast-Slow Chasing Arxiv](https://arxiv.org/abs/2406.05534)
- [Nash-MD Trainer Hugging Face](https://huggingface.co/docs/trl/nash_md_trainer)
- [Nash Learning from Human Feedback Arxiv](https://arxiv.org/pdf/2312.00886)
- [XPO Trainer Hugging Face](https://huggingface.co/docs/trl/xpo_trainer)
- [Exploratory Preference Optimization: Harnessing Implicit Q*-Approximation for Sample-Efficient RLHF Arxiv](https://arxiv.org/abs/2405.21046)
- [Distilling the Knowledge in a Neural Network Arxiv](https://arxiv.org/abs/1503.02531)
- [Everything You Need to Know about Knowledge Distillation Hugging Face](https://huggingface.co/blog/Kseniase/kd)
- [Knowledge Distillation Tutorial PyTorch](https://docs.pytorch.org/tutorials/beginner/knowledge_distillation_tutorial.html)
- [A Comprehensive Survey on Knowledge Distillation Arxiv](https://arxiv.org/abs/2503.12067)
- [Knowledge distillation Wikipedia](https://en.wikipedia.org/wiki/Knowledge_distillation)
- [Generalized Knowledge Distillation Trainer Hugging Face](https://huggingface.co/docs/trl/gkd_trainer)
- [GKD: A General Knowledge Distillation Framework for Large-scale Pre-trained Language Model Arxiv](https://arxiv.org/abs/2306.06629)
- [MiniLLM Trainer Hugging Face](https://huggingface.co/docs/trl/en/minillm_trainer)
- [TRL: Transformer Reinforcement Learning Library -2 Medium](https://medium.com/@danushidk507/trl-transformer-reinforcement-learning-library-2-59186d66ac0b)
- [Learning Task-Agnostic Representations through Multi-Teacher Distillation Arxiv](https://arxiv.org/html/2510.18680v1)
- [Exploring Knowledge Purification in Multi-Teacher Knowledge Distillation for LLMs Openreview](https://openreview.net/forum?id=7pvJoB4aKO)
- [Adaptive Multi-Teacher Knowledge Distillation with Meta-Learning Arxiv](https://arxiv.org/html/2306.06634v1)
- [Knowledge Distillation Tutorial PyTorch](https://docs.pytorch.org/tutorials/beginner/knowledge_distillation_tutorial.html)
- [PILE: Pairwise Iterative Logits Ensemble for Multi-Teacher Labeled Distillation Aclanthology](https://aclanthology.org/2022.emnlp-industry.60.pdf)
- [Towards Cross-Tokenizer Distillation: the Universal Logit Distillation Loss for LLMs Arxiv](https://arxiv.org/html/2402.12030v3)
- [Knowledge Distillation with Refined Logits Arxiv](https://arxiv.org/abs/2408.07703)
- [Enhancing Logits Distillation with Plug&Play Kendall's Ranking Loss Openreview](https://openreview.net/forum?id=msPfUNX9NF)
- [Multi-level Logit Distillation Openaccess](https://openaccess.thecvf.com/content/CVPR2023/papers/Jin_Multi-Level_Logit_Distillation_CVPR_2023_paper.pdf)
- [Self-Distilled Reasoner: On-Policy Self-Distillation for Large Language Models Arxiv](https://arxiv.org/abs/2601.18734)
- [Contrastive Representation Distillation Arxiv](https://arxiv.org/abs/1910.10699)
- [Complementary Relation Contrastive Distillation Openaccess](https://openaccess.thecvf.com/content/CVPR2021/papers/Zhu_Complementary_Relation_Contrastive_Distillation_CVPR_2021_paper.pdf)
- [Contrastive Representation Distillation Openreview](https://openreview.net/forum?id=SkgpBJrtvS)
- [Contrastive Consistent Representation Distillation Bmvc](https://papers.bmvc2023.org/0300.pdf)
- [Improved Knowledge Distillation via Teacher Assistant Arxiv](https://arxiv.org/abs/1902.03393)
- [TAeKD: Teacher Assistant Enhanced Knowledge Distillation for Closed-Source Multilingual Neural Machine Translation Aclanthology](https://aclanthology.org/2024.lrec-main.1350/)
- [A Good Teacher Adapts Their Knowledge for Distillation Openaccess](https://openaccess.thecvf.com/content/ICCV2025/papers/Qian_A_Good_Teacher_Adapts_Their_Knowledge_for_Distillation_ICCV_2025_paper.pdf)
- [Knowledge Distillation using Teacher Assistant Dailydoseofds](https://blog.dailydoseofds.com/p/knowledge-distillation-using-teacher)
- [Meta-learning approaches for few-shot learning: A survey of recent advances Arxiv](https://arxiv.org/abs/2303.07502)
- [Meta-Baseline: Exploring Simple Meta-Learning for Few-Shot Learning CVF](https://openaccess.thecvf.com/content/ICCV2021/papers/Chen_Meta-Baseline_Exploring_Simple_Meta-Learning_for_Few-Shot_Learning_ICCV_2021_paper.pdf)
- [What is Meta-Learning and Few-Shot Learning? Medium](https://medium.com/@myliemudaliyar/pushing-the-limits-of-ai-meta-learning-and-few-shot-learning-2300c69b4c96)
- [Curriculum Meta-Learning for Few-shot Classification Openreview](https://openreview.net/pdf?id=teRUKZdPJt6)
- [A Survey on In-context Learning Arxiv](https://arxiv.org/abs/2301.00234)
- [In Context Learning (ICL) Hopsworks](http://hopsworks.ai/dictionary/in-context-learning-icl)
- [Is In-Context Learning Learning? Arxiv](https://arxiv.org/abs/2509.10414)
- [A Survey on In-context Learning Aclanthology](https://aclanthology.org/2024.emnlp-main.64/)
- [Model-Agnostic Meta-Learning for Fast Adaptation of Deep Networks Arxiv](https://arxiv.org/abs/1703.03400)
- [REPTILE: A Proactive Real-Time Deep Reinforcement Learning Self-adaptive Framework Arxiv](https://arxiv.org/abs/2203.14686)
- [RAMario: Experimental Approach to Reptile Algorithm -- Reinforcement Learning for Mario Arxiv](https://arxiv.org/abs/2305.09655)
- [Reptile: A scalable meta-learning algorithm OpenAI](https://openai.com/index/reptile/)
- [SFT Trainer Hugging Face](https://huggingface.co/docs/trl/en/sft_trainer)
- [Meta in-context learning makes large language models better zero and few-shot relation extractors ACM](https://dl.acm.org/doi/abs/10.24963/ijcai.2024/702)
- [MAML-en-LLM: Model Agnostic Meta-Training of LLMs for Improved In-Context Learning Arxiv](https://arxiv.org/html/2405.11446v1)
- [On the Effectiveness of Adapter-based Tuning for Pretrained Language Model Adaptation Arxiv](https://arxiv.org/abs/2106.03164)
- [Adapters Hugging Face](https://huggingface.co/docs/peft/conceptual_guides/adapter)
- [On the Effectiveness of Adapter-based Tuning for Pretrained Language Model Adaptation Aclanthology](https://aclanthology.org/2021.acl-long.172/)
- [FLoRA: Fused forward-backward adapters for parameter efficient fine-tuning and reducing inference-time latencies of LLMs Arxiv](https://arxiv.org/html/2511.00050v1)
- [Parameter-Efficient Fine-Tuning for Large Models: A Comprehensive Survey Openreview](https://openreview.net/pdf?id=lIsCS8b6zj)
- [Parameter-Efficient Transfer Learning for NLP Arxiv](https://arxiv.org/pdf/1902.00751/1000)
- [Summary Of Adapter Based Performance Efficient Fine Tuning (PEFT) Techniques For Large Language Models Medium](https://siddharth-1729-65206.medium.com/summary-of-adapter-based-performance-efficient-fine-tuning-peft-techniques-for-large-language-fa65d0c2d55f)
- [LLM-Adapters: An Adapter Family for Parameter-Efficient Fine-Tuning of Large Language Models Arxiv](https://arxiv.org/abs/2304.01933)
- [Training Neural Networks from Scratch with Parallel Low-Rank Adapters Openreview](https://openreview.net/forum?id=1SO93f7sVf)
- [Symbiosis: Multi-Adapter Inference and Fine-Tuning Arxiv](https://arxiv.org/abs/2507.03220)
- [Compacter: Efficient Low-Rank Hypercomplex Adapter Layers Arxiv](https://arxiv.org/abs/2106.04647)
- [Fine-Tuning is All You Need: Compact Models Can Outperform GPT's Classification Abilities Researchgate](https://www.researchgate.net/publication/391411130_Fine-Tuning_is_All_You_Need_Compact_Models_Can_Outperform_GPT's_Classification_Abilities)
- [AdaptFormer: Adapting Vision Transformers for Scalable Visual Recognition Arxiv](https://arxiv.org/abs/2205.13535)
- [AdapterFusion: Non-Destructive Task Composition for Transfer Learning Arxiv](https://arxiv.org/abs/2005.00247)
- [AdvFusion: Multilingual Adapter-based Knowledge Transfer for Code Summarization Arxiv](https://arxiv.org/html/2307.07854v2)
- [Pruning Adapterfusion with Lottery Ticket Hypothesis Aclanthology](https://aclanthology.org/2022.findings-naacl.123.pdf)
- [Don't Stop Pretraining? Make Prompt-based Fine-tuning Powerful Learner Arxiv](https://arxiv.org/abs/2305.01711)
- [Fine-tuning after Prompting: an Explainable Way for Classification Aclanthology](https://aclanthology.org/anthology-files/pdf/sighan/2024.sighan-1.16v1.pdf)
- [Is Prompt-Based Finetuning Always Better than Vanilla Finetuning? Insights from Cross-Lingual Language Understanding Arxiv](https://arxiv.org/abs/2307.07880)
- [Prompt Tuning With PEFT Hugging Face](https://huggingface.co/learn/cookbook/prompt_tuning_peft)
- [Prefix-Tuning: Optimizing Continuous Prompts for Generation Arxiv](https://arxiv.org/abs/2101.00190)
- [Prefix tuning Hugging Face](https://huggingface.co/docs/peft/package_reference/prefix_tuning)
- [Prefix-Tuning+: Modernizing Prefix-Tuning by Decoupling the Prefix from Attention Openreview](https://openreview.net/forum?id=GBwx1ej47e)
- [P-Tuning: Prompt Tuning Can Be Comparable to Fine-tuning Across Scales and Tasks Aclanthology](https://aclanthology.org/2022.acl-short.8/)
- [P-Tuning Hugging Face](https://huggingface.co/docs/peft/package_reference/p_tuning)
- [P-Tuning v2: Prompt Tuning Can Be Comparable to Fine-tuning Universally Across Scales and Tasks Arxiv](https://arxiv.org/abs/2110.07602)
- [Ahead-of-Time P-Tuning Openreview](https://openreview.net/forum?id=xTYKdDKS4qG)
- [Fine-Tuning and Prompt Optimization: Two Great Steps that Work Better Together Arxiv](https://arxiv.org/abs/2407.10930)
- [Multitask Prompt Tuning Enables Parameter-Efficient Transfer Learning Arxiv](https://arxiv.org/abs/2303.02861)
- [Introduction to Multi-task Prompt Tuning Heidloff](https://heidloff.net/article/introduction-multi-task-prompt-tuning/)
- [Parameter Efficient Multi-task Fine-tuning by Learning to Transfer Token-wise Prompts Aclanthology](https://aclanthology.org/2023.findings-emnlp.584/)
- [Multitask Prompt Tuning Enables Parameter-Efficient Transfer Learning Openreview](https://openreview.net/forum?id=Nk2pDtuhTq)
- [Learning Optimal Prompt Ensemble for Multi-source Visual Prompt Transfer Arxiv](https://arxiv.org/html/2504.12311v1)
- [Model ensemble instead of prompt fusion: a sample-specific knowledge transfer method for few-shot prompt tuning openreview](https://openreview.net/forum?id=p0yrSRbN5Bu)
- [Efficient Ensemble for Fine-tuning Language Models on Multiple Datasets Arxiv](https://arxiv.org/html/2505.21930v1)
- [DP-Ens DurationQA: Dual-Prompt Fine-Tuning with Log-Probability Ensemble for Duration Question Answering Aclanthology](https://aclanthology.org/2025.vlsp-1.42.pdf)
- [Hybrid approaches to optimization and machine learning methods: a systematic literature review Springer](https://link.springer.com/article/10.1007/s10994-023-06467-x)
- [ReFT: Representation Finetuning for Language Models Arxiv](https://arxiv.org/abs/2404.03592)
- [Enhancing Chain-of-Thought Reasoning with Critical Representation Fine-tuning Aclanthology](https://aclanthology.org/2025.acl-long.1129/)
- [LoReFT | Tune concepts, not weights Medium](https://medium.com/@pushkaraggrawal/loreft-tune-concepts-not-weights-65aed65c6ccd)
- [Speaker Adaptation For Enhancement Of Bone-Conducted Speech IEEE](https://ieeexplore.ieee.org/document/10447322)
- [Parameter efficient speaker adaptation for enhancement of bone-conducted speech PUGS](https://pubs.aip.org/asa/jel/article/6/2/025204/3379901/Parameter-efficient-speaker-adaptation-for)
- [Speaker Adaptation For Enhancement Of Bone-Conducted Speech Illinois](https://experts.illinois.edu/en/publications/speaker-adaptation-for-enhancement-of-bone-conducted-speech/)
- [Scaling & Shifting Your Features: A New Baseline for Efficient Model Tuning Arxiv](https://arxiv.org/abs/2210.08823)
- [TOWARDS A UNIFIED VIEW OF PARAMETER-EFFICIENT TRANSFER LEARNING Arxiv](https://arxiv.org/pdf/2110.04366)
- [Parameter-efficient is not Sufficient: Exploring Parameter, Memory, and Time Efficient Adapter Tuning for Dense Predictions ACM](https://dl.acm.org/doi/10.1145/3664647.3680940)
- [Parameter-efficient transfer learning of pre-trained Transformer models for speaker verification using adapters ResearchGate](https://www.researchgate.net/publication/364932360_Parameter-efficient_transfer_learning_of_pre-trained_Transformer_models_for_speaker_verification_using_adapters)
- [Training with PyTorch](https://docs.pytorch.org/tutorials/beginner/introyt/trainingyt.html)
- [A full training loop Hugging Face](https://huggingface.co/learn/llm-course/chapter3/4)
- [Creating a Training Loop for PyTorch Models Medium](https://medium.com/biased-algorithms/creating-a-training-loop-for-pytorch-models-96e260e70766)
- [Easy JAX training loops with Flax and Optax Hugging Face](https://huggingface.co/blog/afmck/flax-tutorial)
- [Flax](https://flax.readthedocs.io/en/v0.8.1/)
- [Trainer Hugging Face](https://huggingface.co/docs/transformers/en/trainer)
- [Transformers Hugging Face](https://huggingface.co/docs/transformers/en/index)
- [Accelerate Hugging Face](https://huggingface.co/docs/accelerate/index)
- [Introducing 🤗 Accelerate Hugging Face](https://huggingface.co/blog/accelerate-library)
- [Welcome to ⚡ PyTorch Lightning PyTorch](https://lightning.ai/docs/pytorch/stable/)
- [PEFT: Parameter-Efficient Fine-Tuning of Billion-Scale Models on Low-Resource Hardware Hugging Face](https://huggingface.co/blog/peft)
- [Parameter-Efficient Fine-Tuning of Large Language Models for Unit Test Generation: An Empirical Study Arxiv](https://arxiv.org/html/2411.02462)
- [Exploring Parameter-Efficient Fine-Tuning Techniques for Code Generation with Large Language Models ACM](https://dl.acm.org/doi/10.1145/3714461)
- [PEFT Hugging Face](https://huggingface.co/docs/peft/index)
- [LoRA Hugging Face](https://huggingface.co/docs/peft/package_reference/lora)
- [LoRA methods Hugging Face](https://huggingface.co/docs/peft/en/task_guides/lora_based_methods)
- [Implementing LoRA with the PEFT Library Apxml](https://apxml.com/courses/introduction-to-llm-fine-tuning/chapter-4-parameter-efficient-fine-tuning-peft/implementing-lora-with-peft-library)
- [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314)
- [Quantized Low-Rank Adaptation (QLoRA) Apxml](https://apxml.com/courses/fine-tuning-adapting-large-language-models/chapter-4-parameter-efficient-fine-tuning/qlora-introduction)
- [AdaLoRA Hugging Face](https://huggingface.co/docs/peft/package_reference/adalora)
- [AdaLoRA: Adaptive Budget Allocation for Parameter-Efficient Fine-Tuning Arxiv](https://arxiv.org/html/2303.10512v2)
- [VeRA: Vector-based Random Matrix Adaptation Hugging Face](https://huggingface.co/docs/peft/package_reference/vera)
- [VERA: VECTOR-BASED RANDOM MATRIX ADAPTATION Openreview](https://openreview.net/pdf?id=NjNfLdxr3A)
- [Axolotl.ai](https://axolotl.ai/)
- [Fine-Tune ANY Large Language Model (LLM) with Axolotl Medium](https://medium.com/thedeephub/fine-tune-any-large-language-model-llm-with-axolotl-0dc783d52f7e)
- [Quickstart Axolotl](https://docs.axolotl.ai/docs/getting-started.html)
- [Fine-tuning LLMs Guide Unsloth](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide)
- [Orchestrating Distributed Training Jobs Apxml](https://apxml.com/courses/mlops-for-large-models-llmops/chapter-3-llm-training-finetuning-ops/orchestrating-distributed-training)
- [Horovod: fast and easy distributed deep learning in TensorFlow Arxiv](https://arxiv.org/abs/1802.05799)
- [Horovod documentation Horovod](https://horovod.readthedocs.io/en/stable/)
- [Horovod (machine learning) Wikipedia](https://en.wikipedia.org/wiki/Horovod_(machine_learning))
- [Determined Documentation](https://determined.gcp-us1.r.augmentcode.com/docs/index.html)
- [Kubeflow Trainer](https://www.kubeflow.org/docs/components/trainer/)
- [Slurm](https://slurm.schedmd.com/sbatch.html)
- [How to create a Slurm script](https://www.arch.jhu.edu/short-tutorial-how-to-create-a-slurm-script/)